# Conversational RAG with History [Step 2 - Context-Aware Retrieval]

> **MLCourse - Agentic AI - Persistent Memory RAG**

Standard RAG answers each question independently. But real conversations
build on previous turns: "What about her sister?" only makes sense if you
already asked about Alice. This notebook builds a conversational RAG system
that maintains conversation history and uses it to improve retrieval.

### What you will learn

1. Why standalone queries miss conversational context
2. History-aware query rewriting: transforming follow-up questions
3. Building a chain that combines history + retrieval + generation
4. Multi-turn conversations that build on each other
5. Comparing retrieval quality with and without history

### Sections

1. Setup
2. Load and index the document
3. The problem: standalone queries miss context
4. History-aware query rewriting
5. Building the conversational RAG chain
6. Multi-turn conversation demo
7. Comparing retrieval: with vs without history
8. Summary

In [1]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

def _find_track(start_dir):
    here = Path(start_dir).resolve()
    for candidate in (here, *here.parents):
        if candidate.name == "03_agentic_ai":
            return candidate
        if (candidate / "03_agentic_ai").is_dir():
            return candidate / "03_agentic_ai"
    raise FileNotFoundError("Could not locate 03_agentic_ai near %s" % here)

TRACK = _find_track(Path.cwd())
DATA = TRACK / "data"
DATA.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv
load_dotenv(TRACK / ".env", override=False)
load_dotenv(override=False)

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("track:", TRACK)
print("data :", DATA)

track: D:\projects\python\MLCourse\03_agentic_ai
data : D:\projects\python\MLCourse\03_agentic_ai\data


In [2]:
from langchain_ollama import ChatOllama, OllamaEmbeddings

llm = ChatOllama(model="llama3.1:8b", temperature=0)
LLM_LIVE = False
try:
    llm.invoke("Reply with the single word: pong")
    LLM_LIVE = True
    print("[GREEN] Ollama reachable, using llama3.1:8b")
except Exception as exc:
    print("[demo skipped] Ollama not reachable: %s" % type(exc).__name__)
    print("   start Ollama and run: ollama pull llama3.1:8b")
    print("   running with OFFLINE STUB model for this session")

[GREEN] Ollama reachable, using llama3.1:8b


### Section 2: load and index the document


In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_path = str(DATA / "alice.txt")
with open(text_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_text(raw_text)
print("Loaded alice.txt: %d chars -> %d chunks" % (len(raw_text), len(chunks)))


In [4]:
from langchain_chroma import Chroma

embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_texts(chunks, embeddings, collection_name="alice_conv_rag")
print("Vector store built with %d vectors" % vectorstore._collection.count())

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Retriever ready (k=4).")

Vector store built with 191 vectors
Retriever ready (k=4).


### Section 3: the problem -- standalone queries miss context


In [ ]:
# When a user asks a follow-up like "And her cat?", a standalone retriever
# has no idea who "her" refers to. We need to rewrite the query first.
print("=== Standalone Query Problem ===\n")

follow_up = "And what about her cat?"
standalone_docs = retriever.invoke(follow_up)
print("Query: '%s'" % follow_up)
print("Standalone retrieval found %d docs:" % len(standalone_docs))
for i, doc in enumerate(standalone_docs):
    print("  doc %d: %s..." % (i + 1, doc.page_content[:80]))
print()

# With context, the query becomes much clearer
rewritten = "What cat does Alice have in Alice in Wonderland?"
context_docs = retriever.invoke(rewritten)
print("Rewritten query: '%s'" % rewritten)
print("Context-aware retrieval found %d docs:" % len(context_docs))
for i, doc in enumerate(context_docs):
    print("  doc %d: %s..." % (i + 1, doc.page_content[:80]))


### Section 4: history-aware query rewriting


In [ ]:
# A chat model takes the conversation history and the latest user question,
# then produces a standalone query that can be answered without the history.
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given a chat history and the latest user question, formulate a standalone "
     "question that can be understood without the chat history. "
     "Output ONLY the rewritten question, no explanation."),
    ("user", "Chat history:\n{chat_history}\n\nLatest question: {question}\n\nRewritten question:")
])

def rewrite_query(chat_history, question):
    """Rewrite a follow-up question as a standalone query using chat history."""
    # Format history as a simple string
    history_text = ""
    for msg in chat_history:
        role = "User" if msg.type == "human" else "Assistant"
        history_text += "%s: %s\n" % (role, msg.content[:100])

    if LLM_LIVE:
        response = llm.invoke(rewrite_prompt.format_messages(
            chat_history=history_text,
            question=question,
        ))
        return response.content.strip()
    else:
        # Offline stub: simple heuristic rewrite
        if "her" in question.lower() or "his" in question.lower():
            return "Tell me about Alice's cat in Alice in Wonderland"
        return question

# Demo the rewriter
history_sample = [
    type("M", (), {"type": "human", "content": "Tell me about Alice's adventures"})(),
    type("M", (), {"type": "ai", "content": "Alice falls down a rabbit hole into Wonderland..."})(),
]
rewritten = rewrite_query(history_sample, "And what about her cat?")
print("Original follow-up: 'And what about her cat?'")
print("Rewritten query    : '%s'" % rewritten)


### Section 5: building the conversational RAG chain


In [ ]:
# The full chain: rewrite -> retrieve -> generate
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

# Retrieval-augmented generation prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer the user's question using ONLY the "
     "provided context. If the context does not contain enough information, "
     "say so. Be concise and accurate.\n\n"
     "Context:\n{context}"),
    ("user", "{question}")
])

def format_docs(docs):
    """Join retrieved documents into a single context string."""
    return "\n\n---\n\n".join([d.page_content for d in docs])

def conversational_rag(chat_history, question):
    """Full conversational RAG: rewrite -> retrieve -> generate."""
    # Step 1: rewrite the follow-up as a standalone query
    standalone_query = rewrite_query(chat_history, question)

    # Step 2: retrieve relevant documents
    docs = retriever.invoke(standalone_query)
    context = format_docs(docs)

    # Step 3: generate an answer
    messages = rag_prompt.format_messages(context=context, question=question)

    if LLM_LIVE:
        response = llm.invoke(messages)
        answer = response.content
    else:
        # Offline stub: return context preview
        answer = "[offline stub] Based on %d docs (query was rewritten from '%s')" % (
            len(docs), question[:30])
        if docs:
            answer += "\nFirst doc preview: %s..." % docs[0].page_content[:120]

    return {
        "answer": answer,
        "standalone_query": standalone_query,
        "docs": docs,
    }

print("Conversational RAG chain built: rewrite -> retrieve -> generate")


### Section 6: multi-turn conversation demo


In [ ]:
# Simulate a multi-turn conversation that builds on itself.
print("=== Multi-Turn Conversation ===\n")

chat_history = []
turns = [
    "Tell me about Alice's adventures",
    "And what about her cat?",
    "What about the Queen?",
    "How does the story end?",
]

for i, question in enumerate(turns, 1):
    result = conversational_rag(chat_history, question)

    print("Turn %d:" % i)
    print("  user: %s" % question)
    if result["standalone_query"] != question:
        print("  rewritten: '%s'" % result["standalone_query"])
    print("  answer: %s" % result["answer"][:200])
    print("  docs retrieved: %d" % len(result["docs"]))
    print()

    # Update chat history
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=result["answer"]))

print("After %d turns, chat history has %d messages." % (len(turns), len(chat_history)))
print("lesson: each new question builds on the accumulated context.")


### Section 7: comparing retrieval quality


In [ ]:
# Show how history-aware rewriting changes which documents are retrieved.
print("=== Retrieval Comparison ===\n")

test_cases = [
    {
        "question": "What did the rabbit say?",
        "history": [
            AIMessage(content="Alice fell down a rabbit hole into Wonderland."),
        ],
        "context": "follow-up after discussing the rabbit hole",
    },
    {
        "question": "What did the rabbit say?",
        "history": [],
        "context": "standalone (no history)",
    },
]

for tc in test_cases:
    if tc["history"]:
        rewritten = rewrite_query(tc["history"], tc["question"])
    else:
        rewritten = tc["question"]

    docs = retriever.invoke(rewritten)
    print("Context: %s" % tc["context"])
    print("  Original: '%s'" % tc["question"])
    print("  Query sent to retriever: '%s'" % rewritten)
    print("  Retrieved %d docs" % len(docs))
    if docs:
        print("  Top doc preview: %s..." % docs[0].page_content[:100])
    print()

print("lesson: history-aware rewriting produces different (better) queries for follow-ups.")


### Section 8: building a full chain with LCEL


In [ ]:
# For production, use LCEL to compose the rewrite + retrieve + generate steps.
from langchain_core.runnables import RunnablePassthrough

# Build a chain that can be invoked with just {question} and {chat_history}
rewrite_chain = (
    rewrite_prompt
    | llm
    | StrOutputParser()
) if LLM_LIVE else None

if rewrite_chain:
    full_chain = (
        {
            "standalone_query": rewrite_chain,
            "original_question": lambda x: x["question"],
        }
        | {
            "docs": lambda x: retriever.invoke(x["standalone_query"]),
            "question": lambda x: x["original_question"],
        }
        | {
            "context": lambda x: format_docs(x["docs"]),
            "question": lambda x: x["question"],
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    print("LCEL chain built and ready for streaming.")
else:
    print("Offline mode: LCEL chain skipped (requires live LLM).")


In [11]:
print("=" * 60)
print("Conversational RAG Summary:")
print("  Standalone queries lose context from prior turns")
print("  History-aware rewriting transforms follow-ups into standalone queries")
print("  The rewrite -> retrieve -> generate pipeline preserves conversation flow")
print("  More turns accumulate more context, improving answer quality")
print("  LCEL composes the pipeline into a reusable, streamable chain")
print("=" * 60)

Conversational RAG Summary:
  Standalone queries lose context from prior turns
  History-aware rewriting transforms follow-ups into standalone queries
  The rewrite -> retrieve -> generate pipeline preserves conversation flow
  More turns accumulate more context, improving answer quality
  LCEL composes the pipeline into a reusable, streamable chain
